# Understanding TikTok Claims

End-to-end analysis of TikTok claim and opinion videos.


## Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## Load the dataset

In [ ]:
data = pd.read_csv("tiktok_dataset.csv")
data.head()


## Inspect the dataset

In [ ]:
print(f"Rows: {data.shape[0]:,}")
print(f"Columns: {data.shape[1]}")
data.info()


## Check data quality

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": data.dtypes,
    "missing": data.isna().sum(),
    "unique": data.nunique()
})

print(f"Duplicate rows: {data.duplicated().sum()}")
quality_summary


## Review claim status

In [ ]:
data["claim_status"].value_counts(dropna=False)


## Compare engagement by claim status

In [ ]:
engagement_columns = [
    "video_view_count",
    "video_like_count",
    "video_share_count",
    "video_download_count",
    "video_comment_count"
]

data.groupby("claim_status")[engagement_columns].agg(
    ["mean", "median"]
).round(1)


## Visualize claim and opinion videos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=data, x="claim_status", ax=axes[0])
axes[0].set_title("Videos by Claim Status")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")

sns.boxplot(
    data=data,
    x="claim_status",
    y="video_view_count",
    showfliers=False,
    ax=axes[1]
)
axes[1].set_title("Video Views by Claim Status")
axes[1].set_xlabel("")
axes[1].set_ylabel("Views")

plt.tight_layout()
plt.show()


## Examine views and likes

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=data,
    x="video_view_count",
    y="video_like_count",
    hue="claim_status",
    alpha=0.45
)

plt.title("Likes vs. Views by Claim Status")
plt.xlabel("Video Views")
plt.ylabel("Video Likes")
plt.show()


## Compare verification status

In [ ]:
data.groupby("verified_status")["video_view_count"].agg(
    ["count", "mean", "median"]
).round(1)


## Run a Welch t-test

In [ ]:
test_data = data.dropna(
    subset=["verified_status", "video_view_count"]
)

verified_views = test_data.loc[
    test_data["verified_status"] == "verified",
    "video_view_count"
]

unverified_views = test_data.loc[
    test_data["verified_status"] == "not verified",
    "video_view_count"
]

t_stat, p_value = stats.ttest_ind(
    verified_views,
    unverified_views,
    equal_var=False
)

print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.3e}")


## Prepare logistic regression data

In [ ]:
logit_data = data.dropna().drop_duplicates().copy()
logit_data["text_length"] = (
    logit_data["video_transcription_text"].str.len()
)

majority = logit_data[
    logit_data["verified_status"] == "not verified"
]

minority = logit_data[
    logit_data["verified_status"] == "verified"
]

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=RANDOM_STATE
)

logit_data = pd.concat([majority, minority_upsampled])


## Build the logistic regression dataset

In [ ]:
logit_features = [
    "video_duration_sec",
    "claim_status",
    "author_ban_status",
    "video_view_count",
    "video_like_count",
    "video_share_count",
    "video_download_count",
    "video_comment_count",
    "text_length"
]

X_logit = pd.get_dummies(
    logit_data[logit_features],
    drop_first=True
)

y_logit = logit_data["verified_status"].map({
    "not verified": 0,
    "verified": 1
})

X_train_logit, X_test_logit, y_train_logit, y_test_logit = train_test_split(
    X_logit,
    y_logit,
    test_size=0.25,
    stratify=y_logit,
    random_state=RANDOM_STATE
)


## Train logistic regression

In [ ]:
logit_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

logit_model.fit(X_train_logit, y_train_logit)
logit_predictions = logit_model.predict(X_test_logit)

print(classification_report(
    y_test_logit,
    logit_predictions
))


## Evaluate logistic regression

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test_logit,
    logit_predictions
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

logit_coefficients = (
    pd.Series(
        logit_model.coef_[0],
        index=X_train_logit.columns
    )
    .sort_values(key=np.abs, ascending=False)
    .head(10)
)

logit_coefficients.to_frame("coefficient")


## Prepare claim classification data

In [ ]:
model_data = data.dropna().drop_duplicates().copy()

X = model_data.drop(
    columns=[
        "#",
        "video_id",
        "video_transcription_text",
        "claim_status"
    ]
)

X = pd.get_dummies(X, drop_first=True)

y = model_data["claim_status"].map({
    "opinion": 0,
    "claim": 1
})


## Split training, validation, and test data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


## Create an evaluation function

In [ ]:
def evaluate_model(name, model, X_eval, y_eval):
    predictions = model.predict(X_eval)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_eval, predictions),
        "precision": precision_score(y_eval, predictions),
        "recall": recall_score(y_eval, predictions),
        "f1": f1_score(y_eval, predictions)
    }

    print(classification_report(y_eval, predictions))

    ConfusionMatrixDisplay.from_predictions(
        y_eval,
        predictions
    )

    plt.title(f"{name} Confusion Matrix")
    plt.show()

    return metrics


## Tune Random Forest

In [ ]:
random_forest = RandomForestClassifier(
    random_state=RANDOM_STATE
)

random_forest_grid = {
    "max_depth": [5, 7, None],
    "max_features": [0.5],
    "max_samples": [0.7],
    "min_samples_leaf": [1, 2],
    "min_samples_split": [2, 3],
    "n_estimators": [300]
}

scoring = ["accuracy", "precision", "recall", "f1"]

random_forest_search = GridSearchCV(
    random_forest,
    random_forest_grid,
    scoring=scoring,
    cv=5,
    refit="recall"
)

random_forest_search.fit(X_train, y_train)
random_forest_search.best_params_


## Validate Random Forest

In [ ]:
random_forest_results = evaluate_model(
    "Random Forest",
    random_forest_search.best_estimator_,
    X_val,
    y_val
)


## Tune XGBoost

In [ ]:
xgboost = XGBClassifier(
    objective="binary:logistic",
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

xgboost_grid = {
    "max_depth": [4, 6],
    "learning_rate": [0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

xgboost_search = GridSearchCV(
    xgboost,
    xgboost_grid,
    scoring=scoring,
    cv=5,
    refit="recall"
)

xgboost_search.fit(X_train, y_train)
xgboost_search.best_params_


## Validate XGBoost

In [ ]:
xgboost_results = evaluate_model(
    "XGBoost",
    xgboost_search.best_estimator_,
    X_val,
    y_val
)


## Compare model performance

In [ ]:
validation_results = pd.DataFrame([
    random_forest_results,
    xgboost_results
]).set_index("model")

validation_results.round(3)


## Test the champion model

In [ ]:
test_results = evaluate_model(
    "Random Forest",
    random_forest_search.best_estimator_,
    X_test,
    y_test
)


## Review feature importance

In [ ]:
feature_importance = (
    pd.Series(
        random_forest_search
        .best_estimator_
        .feature_importances_,
        index=X.columns
    )
    .sort_values(ascending=False)
    .head(10)
)

feature_importance.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.title("Top Random Forest Predictors")
plt.xlabel("Feature Importance")
plt.ylabel("")
plt.show()


## Key findings

- Claim videos generally received stronger engagement.
- Verified and unverified accounts had significantly different average views.
- Random Forest slightly outperformed XGBoost.
- Views, likes, downloads, shares, and comments were the strongest predictors.
- The model should support human moderation rather than replace it.
